# CLIP + GPT-2 Captioning — Experiment Hub

This notebook is the central place to run, compare, and analyse all experiments for the **M3 ablation suite** and any custom hyperparameter searches.

## What's inside
| Section | Purpose |
|---------|----------|
| **Setup & Config** | One place to change dataset size, LR, epochs, etc. |
| **Run definitions** | The 7 predefined M3 ablation runs |
| **Helper functions** | `run_experiment`, `show_samples`, `plot_training_curve`, `compare_runs` |
| **M3 ablation cells** | One cell per run — execute and inspect inline |
| **Results comparison** | Side-by-side table + bar chart across all 7 runs |
| **Free-form section** | Template for custom hyperparameter experiments |
| **Fine-tuning advice** | What to look for, what to try next |

## Quick-start
1. Set `MAX_SAMPLES = 200` and `DEFAULT_EPOCHS = 1` in **Cell 2** for a ~3-minute smoke test.
2. Once the pipeline works end-to-end, reset `MAX_SAMPLES = -1` (full 31 k images) and run all 7 M3 cells sequentially.
3. Fill in **run_007** after reviewing the results of runs 1–6.



In [ ]:
# ── Colab: mount Drive + authenticate HuggingFace ────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

from google.colab import userdata
import os, logging, sys
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("HuggingFace: authenticated")

# Mirror all logger.info() calls from train.py to notebook stdout
import logging, sys
from transformers import logging as hf_logging

# ── Silence all noisy third-party libraries ───────────────────────
logging.basicConfig(level=logging.WARNING, force=True)
hf_logging.set_verbosity_error()  # hides model load reports + HTTP info

for noisy_lib in [
    "huggingface_hub", "datasets", "urllib3",
    "filelock", "fsspec", "PIL", "httpx",
    "src.preprocessor", "src.data_loader",
]:
    logging.getLogger(noisy_lib).setLevel(logging.ERROR)

# ── Only show our own train.py logs ──────────────────────────────
_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(logging.Formatter("%(message)s"))

for our_logger in ["train", "src.decoder", "src.model"]:
    lg = logging.getLogger(our_logger)
    lg.setLevel(logging.INFO)
    lg.addHandler(_handler)
    lg.propagate = False   # prevent double-printing

print("Logging configured — only training progress will be shown")



In [ ]:
# Cell 1: Setup & Imports
import sys
import json
import argparse
import gc
import random
import warnings
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import torch
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore", category=UserWarning)

# Detect environment
IN_COLAB = "google.colab" in sys.modules
print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    # Clone from GitHub
    import subprocess
    GITHUB_URL = "https://github.com/Scofe-C/Generative_Project_1"

    if not Path("/content/Generative_Project_1").exists():
        subprocess.run(["git", "clone", GITHUB_URL, "/content/Generative_Project_1"], check=True)
    else:
        # Pull latest changes if already cloned
        subprocess.run(["git", "-C", "/content/Generative_Project_1", "pull"], check=True)

    PROJECT_ROOT = Path("/content/Generative_Project_1")

    # Mount Drive just for saving outputs
    from google.colab import drive
    drive.mount("/content/drive")

    # Folder on your Drive where results will be saved
    DRIVE_OUTPUTS = Path("/content/drive/MyDrive/Generative_Project_1_outputs")
    DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

    # Install dependencies
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers", "datasets", "peft",
         "sacrebleu", "pycocoevalcap", "rouge-score", "nltk"],
        check=True,
    )
    print("Cloned + dependencies installed.")

else:
    # Local: walk up to find train.py
    _candidate = Path.cwd()
    for _ in range(4):
        if (_candidate / "train.py").exists():
            break
        _candidate = _candidate.parent
    else:
        raise FileNotFoundError("Could not find project root.")
    PROJECT_ROOT = _candidate
    DRIVE_OUTPUTS = None   # not used locally

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root : {PROJECT_ROOT}")



from train import (
    build_dataset,
    load_models,
    apply_finetuning,
    build_optimiser,
    train_one_epoch,
    evaluate,
    save_checkpoint,
    write_run_outputs,
    make_collate_fn,
    CaptionDataset,
)
from src.utils import get_device, load_config, set_seed, setup_logging

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#  Cell 2: Global Config
# Edit these values to control all experiments.
# For a quick smoke test: set MAX_SAMPLES=200 and DEFAULT_EPOCHS=1.

# Dataset
MAX_SAMPLES = -1       # -1 = full 31 783 images; 200 = smoke test
STRATIFY = False     # False = full dataset (recommended)
                    # True  = 10-class stratified (fast debugging)
SEED = 42

# Training defaults (overridden per run in M3_RUNS / CUSTOM_RUN)
DEFAULT_EPOCHS = 5
DEFAULT_BATCH_SIZE = 64    # A100 40GB handles 64 safely
DEFAULT_LR = 1e-4
MAX_NEW_TOKENS = 50
BEAM_WIDTH = 5

# Evaluation controls
EVAL_EVERY   = 5    # evaluate every N epochs; set to epochs value to only eval last epoch
EVAL_SAMPLES = 500    # -1 = full val set; set 200-500 for fast intermediate feedback

# Output root — always local (download weights manually if needed)
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(parents=True, exist_ok=True)
print(f"Outputs dir  : {OUTPUTS}")

#Load master config and apply dataset overrides
config = load_config()
config["dataset"]["stratify"]  = STRATIFY
config["dataset"]["max_samples"] = MAX_SAMPLES

device = get_device()
set_seed(SEED)

print(f"Device       : {device}")
print(f"Dataset mode : {'full (non-stratified)' if not STRATIFY else '10-class stratified'}")
print(f"Max samples  : {'all' if MAX_SAMPLES < 0 else MAX_SAMPLES}")
print(f"Batch size   : {DEFAULT_BATCH_SIZE} | LR: {DEFAULT_LR} | Epochs: {DEFAULT_EPOCHS}")

In [ ]:
# Cell 3: M3 Ablation Run Definitions
# Each dict maps directly to train.py argument names.
# Keys that are absent here fall back to the defaults in make_args() (Cell 4).
#
# Results from previous round:
#   run_004 (LoRA r=8 + beam) was best: BLEU-4=0.0801, ROUGE-L=0.2210
#   Key findings: LoRA >> frozen, beam >> greedy, ViT-L/14 ~= ViT-B/32, nucleus hurts
#
# This round focuses on:
#   - Caption dropout (already enabled in collate_fn — uses random ref each batch)
#   - Label smoothing (reduces repetitive token overconfidence)
#   - More epochs (previous runs may have underfit)
#   - Higher LR for faster convergence

M3_RUNS = [
    # Run 8 — baseline replay with caption dropout (now built-in)
    dict(
        run_id = "run_008",
        encoder = "openai/clip-vit-base-patch32",
        finetune = "lora",
        decoder = "beam",
        lora_rank = 8,
        epochs = 5,
        lr = 1e-4,
        description = "LoRA r=8 + beam + caption dropout (baseline for round 2)",
    ),
    # Run 9 — label smoothing 0.1
    dict(
        run_id = "run_009",
        encoder = "openai/clip-vit-base-patch32",
        finetune = "lora",
        decoder = "beam",
        lora_rank = 8,
        epochs = 5,
        lr = 1e-4,
        label_smoothing = 0.1,
        description = "LoRA r=8 + beam + label smoothing 0.1",
    ),
    # Run 10 — more epochs (10) to check if model is still learning
    dict(
        run_id = "run_010",
        encoder = "openai/clip-vit-base-patch32",
        finetune = "lora",
        decoder = "beam",
        lora_rank = 8,
        epochs = 10,
        lr = 1e-4,
        label_smoothing = 0.1,
        description = "LoRA r=8 + beam + 10 epochs + label smoothing",
    ),
    # Run 11 — LoRA r=16 sweet spot (between r=8 and r=32)
    dict(
        run_id = "run_011",
        encoder = "openai/clip-vit-base-patch32",
        finetune = "lora",
        decoder = "beam",
        lora_rank = 16,
        epochs = 10,
        lr = 1e-4,
        label_smoothing = 0.1,
        description = "LoRA r=16 + beam + 10 epochs + label smoothing",
    ),
    # Run 12 — higher LR (5e-4) with label smoothing as regularizer
    dict(
        run_id = "run_012",
        encoder = "openai/clip-vit-base-patch32",
        finetune = "lora",
        decoder = "beam",
        lora_rank = 8,
        epochs = 5,
        lr = 5e-4,
        label_smoothing = 0.1,
        description = "LoRA r=8 + beam + higher LR 5e-4 + label smoothing",
    ),
    # Run 13 — ViT-L/14 + all improvements
    dict(
        run_id = "run_013",
        encoder = "openai/clip-vit-large-patch14",
        finetune = "lora",
        decoder = "beam",
        lora_rank = 16,
        epochs = 10,
        lr = 1e-4,
        label_smoothing = 0.1,
        description = "ViT-L/14 + LoRA r=16 + beam + 10ep + label smoothing",
    ),
    # Run 14 — best config from this round (update after runs 8-13)
    dict(
        run_id = "run_014",
        encoder = "openai/clip-vit-base-patch32",
        finetune = "lora",
        decoder = "beam",
        lora_rank = 16,
        epochs = 15,
        lr = 1e-4,
        label_smoothing = 0.1,
        description = "Extended training — 15 epochs with best settings",
    ),
]

print(f"Defined {len(M3_RUNS)} ablation runs:")
for r in M3_RUNS:
    print(f"  {r['run_id']}: {r['description']}")

In [ ]:
# Cell 4: make_args() — run dict → argparse.Namespace
# Converts a run config dict into the Namespace expected by all train.py functions.

def make_args(run_cfg: dict) -> argparse.Namespace:
    """
    Build an argparse.Namespace from a run config dict.
    Any key in run_cfg overrides the corresponding default.
    """
    defaults = dict(
        gpt2 = "gpt2",
        injection = "prefix",
        num_prefix = 10,
        epochs = DEFAULT_EPOCHS,
        batch_size = DEFAULT_BATCH_SIZE,
        lr = DEFAULT_LR,
        max_samples  = MAX_SAMPLES,
        seed = SEED,
        beam_width = BEAM_WIDTH,
        temperature = 1.0,
        top_p = 0.9,
        max_new_tokens = MAX_NEW_TOKENS,
        resume = None,
        label_smoothing = 0.0,
        # Fields required by write_run_outputs / evaluate
        finetune  = "frozen",
        decoder = "greedy",
        lora_rank  = 8,
        encoder  = "openai/clip-vit-base-patch32",
        run_id  = "unnamed",
    )
    merged = {**defaults, **{k: v for k, v in run_cfg.items() if k != "description"}}
    return argparse.Namespace(**merged)

In [ ]:
# Cell 5: run_experiment() — full training loop

def run_experiment(
    run_cfg : dict,
    config  : dict,
    device  : torch.device,
) -> dict:
    """
    Execute a single training experiment defined by run_cfg.

    Calls train.py functions directly (no subprocess) so tensors and logs
    are fully accessible in the notebook.

    Returns a dict with:
        run_id       : str
        scores       : final metric dict {bleu_4, cider, rouge_l, ...}
        hypotheses   : {image_id: [generated_caption]}
        references   : {image_id: [ref1, ref2, ...]}
        train_log    : list of per-epoch dicts
        description  : human-readable run label
    """
    args = make_args(run_cfg)
    desc = run_cfg.get("description", args.run_id)
    print(f"\n{'='*60}")
    print(f"Starting: {args.run_id} — {desc}")
    print(f"  encoder={args.encoder} | finetune={args.finetune}"
          f" | decoder={args.decoder} | lora_rank={args.lora_rank}"
          f" | epochs={args.epochs} | lr={args.lr}")
    print(f"{'='*60}")

    set_seed(args.seed)

    # Clear duplicate log handlers (Colab adds its own; prevent 2x/3x output)
    import logging as _logging
    _logging.getLogger().handlers.clear()
    setup_logging(config)

    run_dir    = OUTPUTS / "runs"    / args.run_id
    weight_dir = OUTPUTS / "weights" / args.run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    weight_dir.mkdir(parents=True, exist_ok=True)

    # 1. Dataset
    print("[1/4] Building dataset and computing CLIP embeddings...")
    # Redirect CLIP cache to Drive so embeddings survive Colab session restarts.
    # Without this, the 20-min CLIP encoding runs fresh every session.
    config["output"]["cache_dir"] = str(OUTPUTS / "cache")
    train_samples, val_samples = build_dataset(args, config, device)
    print(f"      {len(train_samples)} train / {len(val_samples)} val samples")

    # 2. Models
    print("[2/4] Loading models...")
    gpt2_model, prefix_proj, tokenizer = load_models(args, config, device)
    gpt2_model, _ = apply_finetuning(gpt2_model, args)

    # Optionally load CLIP for CLIP-sim metric
    clip_model_eval, clip_proc_eval = None, None
    try:
        from transformers import CLIPModel, CLIPProcessor
        clip_model_eval = CLIPModel.from_pretrained(args.encoder).to(device).eval()
        clip_proc_eval  = CLIPProcessor.from_pretrained(args.encoder)
    except Exception:
        pass

    # 3. Training loop
    print("[3/4] Training...")
    collate_fn   = make_collate_fn(
        tokenizer,
        max_length=config["preprocessing"]["max_token_length"],
    )
    train_loader = DataLoader(
        CaptionDataset(train_samples),
        batch_size  = args.batch_size,
        shuffle     = True,
        collate_fn  = collate_fn,
        num_workers = 0 if not IN_COLAB else min(4, __import__("multiprocessing").cpu_count() // 2),
        pin_memory  = device.type == "cuda",
    )

    n_train_steps            = len(train_loader) * args.epochs
    optimizer, scheduler     = build_optimiser(gpt2_model, prefix_proj, args, n_train_steps)

    best_val_loss             = float("inf")
    train_log: list[dict]     = []
    final_scores: dict        = {}
    final_hypotheses: dict    = {}
    final_references: dict    = {}

    for epoch in range(1, args.epochs + 1):
        print(f"  Epoch {epoch}/{args.epochs}", end="", flush=True)
        train_loss = train_one_epoch(
            epoch, train_loader, gpt2_model, prefix_proj,
            optimizer, scheduler, tokenizer, device,
            label_smoothing=args.label_smoothing,
        )

        is_last_epoch = (epoch == args.epochs)
        if is_last_epoch or (epoch % EVAL_EVERY == 0):
            eval_set = val_samples if EVAL_SAMPLES == -1 else val_samples[:EVAL_SAMPLES]
            scores, hypotheses, references = evaluate(
                eval_set, gpt2_model, prefix_proj, tokenizer, device, args,
                clip_model_eval, clip_proc_eval,
            )
        else:
            scores     = {"bleu_4": 0.0, "cider": 0.0, "rouge_l": 0.0, "meteor": 0.0}
            hypotheses = {}
            references = {}

        val_loss = 1.0 - scores.get("bleu_4", 0.0)
        if scores.get("bleu_4", 0.0) > 0:
            print(f" | train_loss={train_loss:.4f} | BLEU-4={scores.get('bleu_4'):.4f}"
                  f" | CIDEr={scores.get('cider'):.4f} | ROUGE-L={scores.get('rouge_l'):.4f}")
        else:
            print(f" | train_loss={train_loss:.4f} | (eval skipped this epoch)")

        train_log.append({
            "epoch"      : epoch,
            "train_loss" : round(train_loss, 4),
            "bleu_4"     : scores.get("bleu_4",  -1),
            "cider"      : scores.get("cider",   -1),
            "rouge_l"    : scores.get("rouge_l", -1),
            "meteor"     : scores.get("meteor",  -1),
            "lr"         : round(scheduler.get_last_lr()[0], 8),
        })

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss    = val_loss
            final_scores     = scores
            final_hypotheses = hypotheses
            final_references = references

        save_checkpoint(
            weight_dir, epoch, prefix_proj, gpt2_model, val_loss, args, is_best,
        )

    if not final_scores:
        final_scores     = scores
        final_hypotheses = hypotheses
        final_references = references

    # 4. Save outputs
    print("[4/4] Writing run outputs...")
    write_run_outputs(
        run_dir, final_scores, args,
        final_hypotheses, final_references, train_log,
    )

    print(f"\nDone: {args.run_id}")
    print(f"  BLEU-4={final_scores.get('bleu_4', -1):.4f}"
          f" | CIDEr={final_scores.get('cider', -1):.4f}"
          f" | ROUGE-L={final_scores.get('rouge_l', -1):.4f}")

    # Free GPU memory before next run
    del gpt2_model, prefix_proj, train_loader, optimizer, scheduler
    if clip_model_eval is not None:
        del clip_model_eval
    if device.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()


    return {
        "run_id"      : args.run_id,
        "description" : desc,
        "scores"      : final_scores,
        "hypotheses"  : final_hypotheses,
        "references"  : final_references,
        "train_log"   : train_log,
    }

In [ ]:
# Cell 6: show_samples() — qualitative comparison with actual images

def show_samples(result: dict, n: int = 5) -> None:
    """
    Display generated vs reference captions alongside the actual images
    for the first n val samples. Images are streamed from the dataset.
    """
    import matplotlib.gridspec as gridspec
    from datasets import load_dataset
    from IPython.display import display

    run_id      = result["run_id"]
    hypotheses  = result["hypotheses"]
    references  = result["references"]

    # Load per-sample scores from captions.jsonl
    jsonl_path  = OUTPUTS / "runs" / run_id / "captions.jsonl"
    per_sample  = {}
    if jsonl_path.exists():
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                per_sample[rec["image_id"]] = rec

    # Collect target image IDs
    target_ids = set(list(hypotheses.keys())[:n])

    # Stream dataset to find matching images
    ds_cfg = config["dataset"]
    ds = load_dataset(
        ds_cfg["name"],
        split=ds_cfg["split"],
        revision=ds_cfg.get("revision"),
        streaming=True,
    )
    images = {}
    for sample in ds:
        img_id = str(sample.get("image_id", sample.get("img_id", sample.get("id", ""))))
        if img_id in target_ids:
            images[img_id] = sample["image"]
            if len(images) >= len(target_ids):
                break

    print(f"\n{'—'*70}")
    print(f"Sample captions — {run_id}: {result.get('description', '')}")
    print(f"{'—'*70}")

    for img_id in list(hypotheses.keys())[:n]:
        gen_list  = hypotheses[img_id]
        generated = gen_list[0]
        ref_list  = references.get(img_id, [])
        rec       = per_sample.get(img_id, {})
        bleu4     = rec.get("bleu_4", "N/A")
        rouge     = rec.get("rouge_l", "N/A")

        # Build figure: image on left, captions on right
        fig = plt.figure(figsize=(14, 4))
        gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 2], wspace=0.05)

        # Image panel
        ax_img = fig.add_subplot(gs[0])
        if img_id in images:
            ax_img.imshow(images[img_id])
        else:
            ax_img.text(0.5, 0.5, "Image not found", ha="center", va="center",
                        fontsize=12, transform=ax_img.transAxes)
        ax_img.set_title(f"Image {img_id}", fontsize=10, fontweight="bold")
        ax_img.axis("off")

        # Caption panel
        ax_txt = fig.add_subplot(gs[1])
        ax_txt.axis("off")

        score_str = (f"BLEU-4={bleu4:.4f}  ROUGE-L={rouge:.4f}"
                     if isinstance(bleu4, float) else f"BLEU-4={bleu4}  ROUGE-L={rouge}")

        caption_text = f"Generated: {generated}\n\n"
        for i, ref in enumerate(ref_list[:3], 1):
            caption_text += f"Ref {i}: {ref}\n"
        caption_text += f"\n{score_str}"

        ax_txt.text(0.02, 0.95, caption_text, transform=ax_txt.transAxes,
                    fontsize=9, verticalalignment="top", fontfamily="monospace",
                    wrap=True,
                    bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f0f0",
                              edgecolor="#cccccc", alpha=0.9))

        fig.patch.set_facecolor("white")
        fig.tight_layout()
        plt.show()
        plt.close(fig)

    print(f"\n{'—'*70}")
    print(f"Corpus metrics — BLEU-4={result['scores'].get('bleu_4', -1):.4f}"
          f"  CIDEr={result['scores'].get('cider', -1):.4f}"
          f"  ROUGE-L={result['scores'].get('rouge_l', -1):.4f}")

In [ ]:
# Cell 7: plot_training_curve()
def plot_training_curve(train_log: list[dict], run_id: str) -> None:
    """
    Two-panel figure:
      Left  — train_loss over epochs
      Right — BLEU-4, CIDEr, ROUGE-L over epochs
    """
    if not train_log:
        print(f"No training log for {run_id}")
        return

    epochs   = [r["epoch"]      for r in train_log]
    tr_loss  = [r["train_loss"] for r in train_log]
    bleu4    = [r["bleu_4"]     for r in train_log]
    cider    = [r["cider"]      for r in train_log]
    rouge    = [r["rouge_l"]    for r in train_log]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{run_id} — training curves", fontsize=13)

    ax1.plot(epochs, tr_loss, marker="o", color="steelblue", linewidth=2)
    ax1.set_xlabel("Epoch");  ax1.set_ylabel("Train loss")
    ax1.set_title("Training loss")
    ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax1.grid(alpha=0.3)

    ax2.plot(epochs, bleu4, marker="o", label="BLEU-4",  color="#e07b39")
    ax2.plot(epochs, cider, marker="s", label="CIDEr",   color="#4caf50")
    ax2.plot(epochs, rouge, marker="^", label="ROUGE-L", color="#9c27b0")
    ax2.set_xlabel("Epoch");  ax2.set_ylabel("Score")
    ax2.set_title("Validation metrics")
    ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax2.legend();  ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 8: compare_runs() — aggregated metrics table + bar chart

def compare_runs(results: list[dict]) -> pd.DataFrame:
    """
    Build a DataFrame with one row per run showing BLEU-4, CIDEr, ROUGE-L.
    Also renders a grouped bar chart.
    Sorted by CIDEr descending (best model first).
    """
    rows = []
    for r in results:
        if r is None:
            continue
        s = r["scores"]
        rows.append({
            "run_id"      : r["run_id"],
            "description" : r.get("description", ""),
            "BLEU-4"      : round(s.get("bleu_4",  -1), 4),
            "CIDEr"       : round(s.get("cider",   -1), 4),
            "ROUGE-L"     : round(s.get("rouge_l", -1), 4),
            "METEOR"      : round(s.get("meteor",  -1), 4),
        })

    if not rows:
        print("No results to compare yet.")
        return pd.DataFrame()

    df = pd.DataFrame(rows).sort_values("CIDEr", ascending=False).reset_index(drop=True)

    # Bar chart
    metrics  = ["BLEU-4", "CIDEr", "ROUGE-L"]
    n_runs   = len(df)
    x        = np.arange(n_runs)
    width    = 0.22
    colors   = ["#e07b39", "#4caf50", "#9c27b0"]

    fig, ax = plt.subplots(figsize=(max(8, n_runs * 1.4), 5))
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        offsets = x + (i - 1) * width
        bars    = ax.bar(offsets, df[metric], width, label=metric, color=color, alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h >= 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2, h + 0.002,
                    f"{h:.3f}", ha="center", va="bottom", fontsize=7,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(df["run_id"].tolist(), rotation=30, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("M3 Ablation Results — sorted by CIDEr")
    ax.legend();  ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    return df[["run_id", "description", "BLEU-4", "CIDEr", "ROUGE-L", "METEOR"]]

---
## M3 Ablation Runs
Execute the cells below **in order**. Each run saves its outputs to `outputs/runs/{run_id}/`.

> **Tip:** Re-running a cell for a run that already has a cached CLIP embedding file (`outputs/cache/clip_cache_{run_id}.pt`) skips the slow embedding step — subsequent runs of the same `run_id` start immediately.

In [ ]:
#quick check

import torch
from src.decoder import PrefixProjection

# Quick gradient flow test — 5 forward+backward passes
clip_dim  = 512
gpt2_dim  = 768
num_prefix = 10

proj = PrefixProjection(clip_dim, gpt2_dim, num_prefix)
opt  = torch.optim.Adam(proj.parameters(), lr=1e-3)

for step in range(5):
    dummy_clip = torch.randn(4, clip_dim)          # batch of 4
    out = proj(dummy_clip)                          # [4, 10, 768]
    loss = out.pow(2).mean()                        # dummy loss
    loss.backward()
    # projection is nn.Sequential (MLP) — use first Linear layer for grad check
    grad_norm = proj.projection[0].weight.grad.norm().item()
    opt.step(); opt.zero_grad()
    print(f"step {step+1}: loss={loss.item():.4f} | grad_norm={grad_norm:.6f}")

print("If grad_norm > 0 on every step → gradient flow OK")

In [ ]:
# Prefix Similarity Debug Cell
# Run after at least 1 epoch to check if prefix varies per image.
# sim < 0.95 = projection is discriminating images (good)
# sim > 0.99 = projection is degenerate (all images get same caption)
import torch, torch.nn.functional as F

def check_prefix_similarity(prefix_proj, val_samples, device, n_pairs=5):
    prefix_proj.eval()
    print("=== Prefix Cosine Similarity Between Different Images ===")
    print("(< 0.95 = discriminating | > 0.99 = degenerate)")
    with torch.no_grad():
        for i in range(n_pairs):
            a = torch.tensor(val_samples[i]["clip_emb"]).unsqueeze(0).to(device)
            b = torch.tensor(val_samples[i + 50]["clip_emb"]).unsqueeze(0).to(device)
            pa = prefix_proj(a).flatten()
            pb = prefix_proj(b).flatten()
            sim = F.cosine_similarity(pa, pb, dim=0).item()
            status = "OK" if sim < 0.95 else "DEGENERATE"
            print(f"  pair {i+1}: sim={sim:.4f}  [{status}]")

# Usage: run after a training step completes
# check_prefix_similarity(prefix_proj, val_samples, device)
# Or inline after run_experiment:
# check_prefix_similarity(result_001["prefix_proj"], val_samples, device)


### Run 008 — Caption Dropout Baseline
**Question:** Does training with random reference captions (instead of always the first) improve generalization?

**What to look for:**
- Caption dropout is now built into the collate function — every batch picks a random reference.
- Compare directly against run_004 (same config, no caption dropout).
- Expect +1–2 BLEU-4 points from seeing varied phrasing.

In [ ]:
result_008 = run_experiment(M3_RUNS[0], config, device)
show_samples(result_008, n=5)
plot_training_curve(result_008["train_log"], "run_008")

### Run 009 — Label Smoothing
**Question:** Does label smoothing (0.1) reduce repetitive/overconfident token generation?

**What to look for:**
- More diverse vocabulary in captions (fewer repeated phrases).
- Slightly higher train loss (expected — smoothing makes the target softer).
- Better METEOR (sensitive to synonym diversity) even if BLEU-4 is similar.

In [ ]:
result_009 = run_experiment(M3_RUNS[1], config, device)
show_samples(result_009, n=5)
plot_training_curve(result_009["train_log"], "run_009")

### Run 010 — Extended Training (10 epochs)
**Question:** Is the model still improving at epoch 5, or has it converged?

**What to look for:**
- If BLEU-4 keeps climbing after epoch 5, we were undertrained.
- If it plateaus or drops, 5 epochs was enough and more = overfitting.
- Watch train_loss vs val metrics for the divergence point.

In [ ]:
result_010 = run_experiment(M3_RUNS[2], config, device)
show_samples(result_010, n=5)
plot_training_curve(result_010["train_log"], "run_010")

### Run 011 — LoRA r=16 Sweet Spot
**Question:** Is r=16 the sweet spot between r=8 (underpowered) and r=32 (overfitting)?

**What to look for:**
- r=32 didn't help in round 1 — too many params for dataset size.
- r=16 adds moderate capacity. Should outperform r=8 if the model needs it.

In [ ]:
result_011 = run_experiment(M3_RUNS[3], config, device)
show_samples(result_011, n=5)
plot_training_curve(result_011["train_log"], "run_011")

### Run 012 — Higher Learning Rate (5e-4)
**Question:** Is 1e-4 too conservative? Does 5x higher LR converge faster?

**What to look for:**
- Faster initial loss drop (first 2 epochs).
- Risk of instability — if loss spikes, LR is too high.
- Label smoothing acts as regularizer, making higher LR safer.

In [ ]:
result_012 = run_experiment(M3_RUNS[4], config, device)
show_samples(result_012, n=5)
plot_training_curve(result_012["train_log"], "run_012")

### Run 013 — ViT-L/14 + All Improvements
**Question:** Does ViT-L/14 benefit more from caption dropout + label smoothing + longer training?

**What to look for:**
- Round 1 showed ViT-L/14 didn't help. Maybe it needed more epochs and better regularization.
- If it still doesn't help, ViT-B/32 is sufficient for this dataset.

In [ ]:
result_013 = run_experiment(M3_RUNS[5], config, device)
show_samples(result_013, n=5)
plot_training_curve(result_013["train_log"], "run_013")

### Run 014 — Extended Best Config (15 epochs)
**Question:** How far can we push the best settings with more training?

**Before running:** Update run_014 in Cell 3 with the best encoder/rank/LR from runs 008–013.

In [ ]:
result_014 = run_experiment(M3_RUNS[6], config, device)
show_samples(result_014, n=5)
plot_training_curve(result_014["train_log"], "run_014")

---
## Results Comparison
Run this cell after all 7 ablation runs are complete. The table is sorted by CIDEr (best model first).

In [ ]:
# Collect whichever results exist
all_results = [
    result_008 if 'result_008' in dir() else None,
    result_009 if 'result_009' in dir() else None,
    result_010 if 'result_010' in dir() else None,
    result_011 if 'result_011' in dir() else None,
    result_012 if 'result_012' in dir() else None,
    result_013 if 'result_013' in dir() else None,
    result_014 if 'result_014' in dir() else None,
]
all_results = [r for r in all_results if r is not None]

if all_results:
    df = compare_runs(all_results)
    display(df)
else:
    print("No results yet. Run the experiment cells above first.")

---
## Free-form Custom Experiments
Use the template below to run any custom hyperparameter combination.
Change `run_id` for each new experiment to avoid overwriting previous results.

In [ ]:
# Custom Experiment Template
# Modify any of the parameters below and execute this cell.
# Each run is saved independently — compare with compare_runs() afterwards.

CUSTOM_RUN = dict(
    run_id      = "custom_001",                        # change for each new run
    description = "Custom: LoRA r=16, 5 epochs, lr=1e-4",

    # Model config
    encoder     = "openai/clip-vit-base-patch32",      # or clip-vit-large-patch14
    finetune    = "lora",                              # frozen | lora | prefix_tuning
    lora_rank   = 16,                                  # try: 4, 8, 16, 32
    decoder     = "beam",                              # greedy | beam | nucleus
    injection   = "prefix",                            # prefix | cross_attention
    num_prefix  = 10,                                  # try: 5, 10, 20, 30

    # Training config
    epochs      = 5,                                   # try: 3, 5, 8, 10
    lr          = 1e-4,                                # try: 1e-5, 5e-5, 1e-4, 2e-4
    batch_size  = 32,                                  # A100: try 32, 48, 64
)

custom_result = run_experiment(CUSTOM_RUN, config, device)
show_samples(custom_result, n=5)
plot_training_curve(custom_result["train_log"], CUSTOM_RUN["run_id"])

# Optionally compare to prior runs:
# compare_runs([result_004, custom_result])

---
## Fine-tuning Advice & Experiment Guide

### Recommended experiment order
1. **run_001 (frozen baseline)** — confirm the pipeline works, establish the floor.
2. **run_002 (LoRA r=8)** — biggest single improvement; should clearly beat frozen.
3. **run_004 (beam search)** — free BLEU gain with no additional training.
4. **run_005 (ViT-L/14)** — check if a larger encoder meaningfully improves CIDEr.
5. **Free-form sweep** — vary LoRA rank, epochs, LR once you know the best base config.
6. **run_007** — reproduce the best config for final reported numbers.

---

### What metrics to optimise for
| Metric | Best for | Typical range |
|--------|----------|--------------|
| **CIDEr** | Primary captioning metric, best human correlation | 0.3–1.2 |
| **BLEU-4** | Most commonly reported in papers, strict n-gram | 0.05–0.25 |
| **ROUGE-L** | Fluency / longest common subsequence | 0.2–0.5 |

→ **Optimise for CIDEr** when comparing architectures. Use BLEU-4 for paper reporting.

---

### Overfitting signals (watch for these)
- Train loss drops but val BLEU-4 plateaus or declines after epoch 2.
- Generated captions look fluent but are generic ("a man and a woman are standing outside").

**Fixes:**
- Reduce LoRA rank (`r=4` instead of `r=8`)
- Reduce epochs (3 → 2, use best checkpoint)
- Reduce LR (`5e-5` → `1e-5`)
- Add LoRA dropout (`lora_dropout=0.1`) — currently at 0.05

---

### Hyperparameter sweep guide

| Parameter | Conservative | Aggressive | Notes |
|-----------|-------------|------------|-------|
| LoRA rank | 4 | 16–32 | Higher = more params, higher overfit risk on small data |
| Prefix tokens K | 5 | 20–30 | More capacity but slower convergence |
| Epochs | 3 | 8–10 | Stop after 2 non-improving epochs |
| Learning rate | 1e-5 | 2e-4 | Cosine schedule handles decay |
| Beam width | 3 | 10 | Diminishing returns above 5 for short captions |
| Batch size | 16 | 64 (A100) | Larger batch → lower LR may be needed |

---

### Recommended custom sweeps after M3

```python
# Sweep 1: LoRA rank
for rank in [4, 8, 16, 32]:
    run_experiment(dict(run_id=f"lora_r{rank}", finetune="lora",
                        lora_rank=rank, decoder="beam",
                        encoder="openai/clip-vit-large-patch14"), config, device)

# Sweep 2: Prefix tokens K
for k in [5, 10, 20, 30]:
    run_experiment(dict(run_id=f"prefix_k{k}", finetune="lora",
                        num_prefix=k, decoder="beam",
                        encoder="openai/clip-vit-large-patch14"), config, device)

# Sweep 3: Beam width
for bw in [1, 3, 5, 10]:
    run_experiment(dict(run_id=f"beam_w{bw}", finetune="lora",
                        decoder="beam", beam_width=bw,
                        encoder="openai/clip-vit-large-patch14"), config, device)
```